# 🤖 Humanoid Ball RL — Curriculum Learning

## Curriculum stages:
| Phase | Name | Task |
|-------|------|------|
| 1 | **Stability** | Stand upright and stay balanced |
| 2 | **Touch** | Approach and make foot contact with the ball |
| 3 | **Pass** | Kick the ball ~3 m so it stops there |
| 4 | **Two Robots** | Two humanoids alternate passing the ball |

**Stack**: MuJoCo 3.x · Stable-Baselines3 PPO · Custom CurriculumCallback

## 📦 Cell 1 – Install

In [ ]:
%%bash
pip install -q mujoco stable-baselines3[extra] gymnasium imageio mediapy
apt-get install -q -y ffmpeg libgl1-mesa-glx libosmesa6-dev xvfb
echo 'Done'


In [ ]:
import subprocess, os
# Start Xvfb for headless rendering in Colab / remote environments
os.makedirs('/tmp/mujoco_tmp', exist_ok=True)
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x720x24', '-ac'])
os.environ['DISPLAY'] = ':99'
os.environ['MUJOCO_GL'] = 'osmesa'
print("Xvfb started, MUJOCO_GL=osmesa")


## 🏗️ Cell 2 – Single-Robot XML

In [ ]:
SINGLE_XML = '''
<mujoco model="humanoid_ball">
  <compiler angle="radian" autolimits="true"/>
  <option timestep="0.005" gravity="0 0 -9.81" iterations="50"
          solver="Newton" tolerance="1e-10"/>
  <default>
    <joint limited="true" damping="0.1" armature="0.01"/>
    <geom contype="1" conaffinity="1" condim="3" friction="0.8 0.1 0.1"/>
  </default>
  <asset>
    <material name="mat_grey"  rgba="0.6 0.6 0.6 1"/>
    <material name="mat_blue"  rgba="0.2 0.4 0.8 1"/>
    <material name="mat_red"   rgba="0.8 0.2 0.2 1"/>
    <material name="mat_white" rgba="0.9 0.9 0.9 1"/>
    <material name="mat_green" rgba="0.1 0.7 0.1 1"/>
    <material name="mat_floor" rgba="0.5 0.5 0.5 1" reflectance="0.3"/>
  </asset>
  <worldbody>
    <geom name="floor" type="plane" size="10 10 0.1" material="mat_floor" friction="0.8 0.1 0.1"/>
    <body name="torso" pos="0 0 1.25">
      <freejoint name="root"/>
      <geom name="torso_geom" type="capsule" size="0.07 0.18" material="mat_blue"/>
      <site name="torso_site" pos="0 0 0" size="0.02"/>
      <body name="head" pos="0 0 0.27">
        <geom type="sphere" size="0.09" material="mat_blue"/>
      </body>
      <body name="abdomen" pos="0 0 -0.22">
        <joint name="abdomen_y" type="hinge" axis="0 1 0" range="-0.5 0.5"/>
        <joint name="abdomen_z" type="hinge" axis="0 0 1" range="-0.5 0.5"/>
        <geom type="capsule" size="0.06 0.10" material="mat_blue"/>
        <body name="right_thigh" pos="0.09 0 -0.16">
          <joint name="right_hip_x" type="hinge" axis="1 0 0" range="-1.0 1.0"/>
          <joint name="right_hip_z" type="hinge" axis="0 0 1" range="-1.0 0.5"/>
          <joint name="right_hip_y" type="hinge" axis="0 1 0" range="-1.7 0.5" damping="2.0"/>
          <geom type="capsule" size="0.05 0.17" quat="1 0 0.5 0" material="mat_red"/>
          <body name="right_shin" pos="0.01 0 -0.38">
            <joint name="right_knee" type="hinge" axis="0 -1 0" range="0.05 2.5" damping="3.0"/>
            <geom type="capsule" size="0.04 0.15" quat="1 0 0.08 0" material="mat_red"/>
            <body name="right_foot" pos="0.01 0 -0.33">
              <joint name="right_ankle_y" type="hinge" axis="0 1 0" range="-1.0 0.5" damping="2.0"/>
              <joint name="right_ankle_x" type="hinge" axis="1 0 0" range="-0.5 0.8" damping="2.0"/>
              <geom type="capsule" size="0.03" fromto="-0.05 -0.02 0 0.15 -0.02 0" material="mat_grey"/>
              <geom type="capsule" size="0.03" fromto="-0.05  0.02 0 0.15  0.02 0" material="mat_grey"/>
              <site name="right_foot_site" pos="0.1 0 0" size="0.02"/>
            </body>
          </body>
        </body>
        <body name="left_thigh" pos="-0.09 0 -0.16">
          <joint name="left_hip_x" type="hinge" axis="-1 0 0" range="-1.0 1.0"/>
          <joint name="left_hip_z" type="hinge" axis="0 0 -1" range="-1.0 0.5"/>
          <joint name="left_hip_y" type="hinge" axis="0 1 0" range="-1.7 0.5" damping="2.0"/>
          <geom type="capsule" size="0.05 0.17" quat="1 0 -0.5 0" material="mat_red"/>
          <body name="left_shin" pos="-0.01 0 -0.38">
            <joint name="left_knee" type="hinge" axis="0 -1 0" range="0.05 2.5" damping="3.0"/>
            <geom type="capsule" size="0.04 0.15" quat="1 0 -0.08 0" material="mat_red"/>
            <body name="left_foot" pos="-0.01 0 -0.33">
              <joint name="left_ankle_y" type="hinge" axis="0 1 0" range="-1.0 0.5" damping="2.0"/>
              <joint name="left_ankle_x" type="hinge" axis="1 0 0" range="-0.5 0.8" damping="2.0"/>
              <geom type="capsule" size="0.03" fromto="-0.05 -0.02 0 0.15 -0.02 0" material="mat_grey"/>
              <geom type="capsule" size="0.03" fromto="-0.05  0.02 0 0.15  0.02 0" material="mat_grey"/>
              <site name="left_foot_site" pos="0.1 0 0" size="0.02"/>
            </body>
          </body>
        </body>
      </body>
      <body name="right_upper_arm" pos="0.16 0 0.13">
        <joint name="right_shoulder1" type="hinge" axis="2 1 1" range="-2.8 2.8"/>
        <joint name="right_shoulder2" type="hinge" axis="0 -1 1" range="-3.0 2.5"/>
        <geom type="capsule" size="0.04 0.10" quat="1 1 0 0" material="mat_blue"/>
        <body name="right_lower_arm" pos="0.18 0 -0.02">
          <joint name="right_elbow" type="hinge" axis="0 -1 1" range="-2.0 0.2" damping="1.0"/>
          <geom type="capsule" size="0.031 0.09" quat="1 1 0 0" material="mat_blue"/>
          <body name="right_hand" pos="0.16 0 -0.01">
            <geom type="sphere" size="0.04" material="mat_grey"/>
          </body>
        </body>
      </body>
      <body name="left_upper_arm" pos="-0.16 0 0.13">
        <joint name="left_shoulder1" type="hinge" axis="2 -1 1" range="-2.8 2.8"/>
        <joint name="left_shoulder2" type="hinge" axis="0 1 1" range="-3.0 2.5"/>
        <geom type="capsule" size="0.04 0.10" quat="1 1 0 0" material="mat_blue"/>
        <body name="left_lower_arm" pos="-0.18 0 -0.02">
          <joint name="left_elbow" type="hinge" axis="0 -1 -1" range="-2.0 0.2" damping="1.0"/>
          <geom type="capsule" size="0.031 0.09" quat="1 1 0 0" material="mat_blue"/>
          <body name="left_hand" pos="-0.16 0 -0.01">
            <geom type="sphere" size="0.04" material="mat_grey"/>
          </body>
        </body>
      </body>
    </body>
    <body name="ball" pos="1.0 0 0.08">
      <freejoint name="ball_joint"/>
      <geom name="ball_geom" type="sphere" size="0.08" material="mat_white" mass="0.15"
            friction="0.6 0.1 0.1" solimp="0.95 0.99 0.001" solref="0.015 1"/>
      <site name="ball_site" pos="0 0 0" size="0.015"/>
    </body>
    <body name="target_marker" pos="3.0 0 0.01" mocap="true">
      <geom type="cylinder" size="0.15 0.005" material="mat_green" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <actuator>
    <motor name="abdomen_y"      joint="abdomen_y"      gear="40"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="abdomen_z"      joint="abdomen_z"      gear="40"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_hip_x"    joint="right_hip_x"    gear="80"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_hip_z"    joint="right_hip_z"    gear="80"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_hip_y"    joint="right_hip_y"    gear="120" ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_hip_x"     joint="left_hip_x"     gear="80"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_hip_z"     joint="left_hip_z"     gear="80"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_hip_y"     joint="left_hip_y"     gear="120" ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_knee"     joint="right_knee"     gear="100" ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_knee"      joint="left_knee"      gear="100" ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_ankle_y"  joint="right_ankle_y"  gear="40"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_ankle_x"  joint="right_ankle_x"  gear="20"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_ankle_y"   joint="left_ankle_y"   gear="40"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_ankle_x"   joint="left_ankle_x"   gear="20"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_shoulder1" joint="right_shoulder1" gear="20" ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_shoulder2" joint="right_shoulder2" gear="20" ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="right_elbow"    joint="right_elbow"    gear="20"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_shoulder1" joint="left_shoulder1" gear="20"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_shoulder2" joint="left_shoulder2" gear="20"  ctrllimited="true" ctrlrange="-1 1"/>
    <motor name="left_elbow"     joint="left_elbow"     gear="20"  ctrllimited="true" ctrlrange="-1 1"/>
  </actuator>
  <sensor>
    <accelerometer name="torso_accel" site="torso_site"/>
    <gyro          name="torso_gyro"  site="torso_site"/>
    <touch         name="right_foot_touch" site="right_foot_site"/>
    <touch         name="left_foot_touch"  site="left_foot_site"/>
    <touch         name="ball_touch"       site="ball_site"/>
  </sensor>
</mujoco>
'''

import os
os.makedirs('models', exist_ok=True)
with open('models/humanoid_ball.xml', 'w') as f:
    f.write(SINGLE_XML)
print('Single robot XML saved')


## 🤖🤖 Cell 3 – Two-Robot XML

In [ ]:
def _make_leg(prefix, side, sign):
    s = '' if sign > 0 else '-'
    return f'''
        <body name="{prefix}_{side}_thigh" pos="{sign*0.09} 0 -0.16">
          <joint name="{prefix}_{side}_hip_x" type="hinge" axis="{sign} 0 0" range="-1.0 1.0"/>
          <joint name="{prefix}_{side}_hip_z" type="hinge" axis="0 0 {sign}" range="-1.0 0.5"/>
          <joint name="{prefix}_{side}_hip_y" type="hinge" axis="0 1 0" range="-1.7 0.5" damping="2.0"/>
          <geom type="capsule" size="0.05 0.17" quat="1 0 {sign*0.5} 0" material="mat_grey"/>
          <body name="{prefix}_{side}_shin" pos="{sign*0.01} 0 -0.38">
            <joint name="{prefix}_{side}_knee" type="hinge" axis="0 -1 0" range="0.05 2.5" damping="3.0"/>
            <geom type="capsule" size="0.04 0.15" quat="1 0 {sign*0.08} 0" material="mat_grey"/>
            <body name="{prefix}_{side}_foot" pos="{sign*0.01} 0 -0.33">
              <joint name="{prefix}_{side}_ankle_y" type="hinge" axis="0 1 0" range="-1.0 0.5" damping="2.0"/>
              <joint name="{prefix}_{side}_ankle_x" type="hinge" axis="1 0 0" range="-0.5 0.8" damping="2.0"/>
              <geom type="capsule" size="0.03" fromto="-0.05 -0.02 0 0.15 -0.02 0" material="mat_grey"/>
              <geom type="capsule" size="0.03" fromto="-0.05  0.02 0 0.15  0.02 0" material="mat_grey"/>
              <site name="{prefix}_{side}_foot_site" pos="0.1 0 0" size="0.02"/>
            </body>
          </body>
        </body>'''

def _make_arm(prefix, side, sign):
    return f'''
      <body name="{prefix}_{side}_upper_arm" pos="{sign*0.16} 0 0.13">
        <joint name="{prefix}_{side}_shoulder1" type="hinge" axis="2 {sign} 1" range="-2.8 2.8"/>
        <joint name="{prefix}_{side}_shoulder2" type="hinge" axis="0 {-sign} 1" range="-3.0 2.5"/>
        <geom type="capsule" size="0.04 0.10" quat="1 1 0 0" material="mat_grey"/>
        <body name="{prefix}_{side}_lower_arm" pos="{sign*0.18} 0 -0.02">
          <joint name="{prefix}_{side}_elbow" type="hinge" axis="0 -1 {sign}" range="-2.0 0.2" damping="1.0"/>
          <geom type="capsule" size="0.031 0.09" quat="1 1 0 0" material="mat_grey"/>
        </body>
      </body>'''

def make_robot_body(prefix, x_pos, mat):
    legs  = _make_leg(prefix, 'right', 1) + _make_leg(prefix, 'left', -1)
    arms  = _make_arm(prefix, 'right', 1) + _make_arm(prefix, 'left', -1)
    return f'''
    <body name="{prefix}_torso" pos="{x_pos} 0 1.25">
      <freejoint name="{prefix}_root"/>
      <geom type="capsule" size="0.07 0.18" material="{mat}"/>
      <site name="{prefix}_torso_site" pos="0 0 0" size="0.02"/>
      <body name="{prefix}_head" pos="0 0 0.27">
        <geom type="sphere" size="0.09" material="{mat}"/>
      </body>
      <body name="{prefix}_abdomen" pos="0 0 -0.22">
        <joint name="{prefix}_abdomen_y" type="hinge" axis="0 1 0" range="-0.5 0.5"/>
        <joint name="{prefix}_abdomen_z" type="hinge" axis="0 0 1" range="-0.5 0.5"/>
        <geom type="capsule" size="0.06 0.10" material="{mat}"/>
        {legs}
      </body>
      {arms}
    </body>'''

def make_actuators(prefix):
    joints = [
        ('abdomen_y',40),('abdomen_z',40),
        ('right_hip_x',80),('right_hip_z',80),('right_hip_y',120),
        ('left_hip_x',80), ('left_hip_z',80), ('left_hip_y',120),
        ('right_knee',100),('left_knee',100),
        ('right_ankle_y',40),('right_ankle_x',20),
        ('left_ankle_y',40), ('left_ankle_x',20),
        ('right_shoulder1',20),('right_shoulder2',20),('right_elbow',20),
        ('left_shoulder1',20), ('left_shoulder2',20), ('left_elbow',20),
    ]
    return '\n    '.join(
        f'<motor name="{prefix}_{j}" joint="{prefix}_{j}" gear="{g}" ctrllimited="true" ctrlrange="-1 1"/>'
        for j,g in joints
    )

two_xml = f'''<mujoco model="two_humanoids_ball">
  <compiler angle="radian" autolimits="true"/>
  <option timestep="0.005" gravity="0 0 -9.81" iterations="50" solver="Newton" tolerance="1e-10"/>
  <default>
    <joint limited="true" damping="0.1" armature="0.01"/>
    <geom contype="1" conaffinity="1" condim="3" friction="0.8 0.1 0.1"/>
  </default>
  <asset>
    <material name="mat_grey"  rgba="0.6 0.6 0.6 1"/>
    <material name="mat_blue"  rgba="0.2 0.4 0.8 1"/>
    <material name="mat_red"   rgba="0.8 0.2 0.2 1"/>
    <material name="mat_white" rgba="0.9 0.9 0.9 1"/>
    <material name="mat_green" rgba="0.1 0.7 0.1 1"/>
    <material name="mat_floor" rgba="0.5 0.5 0.5 1" reflectance="0.3"/>
  </asset>
  <worldbody>
    <geom name="floor" type="plane" size="20 20 0.1" material="mat_floor" friction="0.8 0.1 0.1"/>
    {make_robot_body('robot1', -3.0, 'mat_blue')}
    {make_robot_body('robot2',  3.0, 'mat_red')}
    <body name="ball" pos="-2.0 0 0.08">
      <freejoint name="ball_joint"/>
      <geom name="ball_geom" type="sphere" size="0.08" material="mat_white" mass="0.15"
            friction="0.6 0.1 0.1" solimp="0.95 0.99 0.001" solref="0.015 1"/>
      <site name="ball_site" pos="0 0 0" size="0.015"/>
    </body>
    <body name="target_marker" pos="3.0 0 0.01" mocap="true">
      <geom type="cylinder" size="0.2 0.005" material="mat_green" contype="0" conaffinity="0"/>
    </body>
  </worldbody>
  <actuator>
    {make_actuators('robot1')}
    {make_actuators('robot2')}
  </actuator>
  <sensor>
    <accelerometer name="robot1_torso_accel" site="robot1_torso_site"/>
    <gyro          name="robot1_torso_gyro"  site="robot1_torso_site"/>
    <touch         name="robot1_right_foot_touch" site="robot1_right_foot_site"/>
    <touch         name="robot1_left_foot_touch"  site="robot1_left_foot_site"/>
    <accelerometer name="robot2_torso_accel" site="robot2_torso_site"/>
    <gyro          name="robot2_torso_gyro"  site="robot2_torso_site"/>
    <touch         name="robot2_right_foot_touch" site="robot2_right_foot_site"/>
    <touch         name="robot2_left_foot_touch"  site="robot2_left_foot_site"/>
    <touch         name="ball_touch"              site="ball_site"/>
  </sensor>
</mujoco>'''

import os
os.makedirs('models', exist_ok=True)
with open('models/two_humanoids_ball.xml', 'w') as f:
    f.write(two_xml)
print('Two-robot XML saved')


import numpy as np
import mujoco
import gymnasium as gym
from gymnasium import spaces

N_ACT_SINGLE   = 20
# Single-robot obs layout: qpos(34) + qvel(32) + sensors(9) + ball_rel_pos(3) + ball_vel(3) = 81
OBS_DIM_SINGLE = 81
PASS_DISTANCE  = 3.0
BALL_STOP_VEL  = 0.05
HEIGHT_TARGET  = 1.25

class HumanoidBallEnv(gym.Env):
    """
    Single humanoid + ball.
    phase: 1=Stability  2=Touch  3=Pass
    """
    metadata = {"render_modes": ["rgb_array"], "render_fps": 30}

    def __init__(self, phase=1, render_mode=None):
        super().__init__()
        self.phase = phase
        self.render_mode = render_mode
        self.model = mujoco.MjModel.from_xml_path("models/humanoid_ball.xml")
        self.data  = mujoco.MjData(self.model)

        obs_hi = np.full(OBS_DIM_SINGLE, np.inf, dtype=np.float32)
        self.observation_space = spaces.Box(-obs_hi, obs_hi)
        self.action_space      = spaces.Box(-1.0, 1.0, shape=(N_ACT_SINGLE,), dtype=np.float32)

        self.max_steps    = 1000
        self._step_count  = 0
        self._ball_kicked = False
        self._kick_step   = 0
        self._renderer    = None

        def bid(n): return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY,  n)
        def jid(n): return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, n)
        self._torso_id    = bid("torso")
        self._ball_id     = bid("ball")
        self._rfoot_id    = bid("right_foot")
        self._lfoot_id    = bid("left_foot")
        self._ball_jnt_id = jid("ball_joint")

    def _torso_pos(self): return self.data.xpos[self._torso_id].copy()
    def _ball_pos(self):  return self.data.xpos[self._ball_id].copy()
    def _ball_vel(self):
        va = self.model.jnt_dofadr[self._ball_jnt_id]
        return self.data.qvel[va:va+3].copy()

    def _get_obs(self):
        tp   = self._torso_pos()
        bp   = self._ball_pos()
        bv   = self._ball_vel()
        # qpos=34, qvel=32, sensors=9, bp-tp=3, bv=3  → total 81
        raw = np.concatenate([
            self.data.qpos,
            self.data.qvel,
            self.data.sensordata,
            bp - tp,
            bv,
        ]).astype(np.float32)
        if len(raw) < OBS_DIM_SINGLE:
            raw = np.pad(raw, (0, OBS_DIM_SINGLE - len(raw)))
        return raw[:OBS_DIM_SINGLE]

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] += self.np_random.uniform(-0.01, 0.01, self.model.nq)
        ball_x = {1: 5.0, 2: 0.8, 3: 0.7}.get(self.phase, 0.7)
        qa = self.model.jnt_qposadr[self._ball_jnt_id]
        self.data.qpos[qa]     = ball_x
        self.data.qpos[qa+1]   = 0.0
        self.data.qpos[qa+2]   = 0.08
        self.data.qpos[qa+3:qa+7] = [1, 0, 0, 0]
        mujoco.mj_forward(self.model, self.data)
        self._step_count  = 0
        self._ball_kicked = False
        self._kick_step   = 0
        return self._get_obs(), {}

    def step(self, action):
        self.data.ctrl[:] = np.clip(action, -1, 1)
        for _ in range(4): mujoco.mj_step(self.model, self.data)
        self._step_count += 1
        obs   = self._get_obs()
        rew   = self._compute_reward()
        term  = self._is_terminated()
        trunc = self._step_count >= self.max_steps
        return obs, rew, term, trunc, {"phase": self.phase}

    def _compute_reward(self):
        tp   = self._torso_pos()
        bp   = self._ball_pos()
        bv   = self._ball_vel()
        bspd = np.linalg.norm(bv)
        ctrl_pen = 0.001 * np.sum(self.data.ctrl ** 2)

        # Phase 1 – Stability
        if self.phase == 1:
            h_err   = abs(tp[2] - HEIGHT_TARGET)
            upright = np.exp(-4 * h_err)
            alive   = 1.0 if tp[2] > 0.8 else -2.0
            return alive + upright - ctrl_pen

        # Phase 2 – Touch
        elif self.phase == 2:
            if tp[2] < 0.8: return -3.0
            rf = self.data.xpos[self._rfoot_id]
            lf = self.data.xpos[self._lfoot_id]
            fd       = min(np.linalg.norm(rf - bp), np.linalg.norm(lf - bp))
            approach = np.exp(-2 * np.linalg.norm(tp[:2] - bp[:2]))
            touch    = np.exp(-5 * fd)
            moved    = 2.0 if bspd > 0.3 else 0.0
            return 0.5 * approach + touch + moved - ctrl_pen

        # Phase 3 – Pass
        else:
            if tp[2] < 0.8: return -3.0
            if not self._ball_kicked and bspd > 1.0:
                self._ball_kicked = True
                self._kick_step   = self._step_count
            if self._ball_kicked:
                dist_err = abs(bp[0] - PASS_DISTANCE)
                dist_r   = np.exp(-1.5 * dist_err)
                stop_r   = 5.0 if bspd < BALL_STOP_VEL else 0.0
                lat_pen  = 0.5 * abs(bp[1])
                return dist_r + stop_r - lat_pen - ctrl_pen
            else:
                rf = self.data.xpos[self._rfoot_id]
                fd = np.linalg.norm(rf - bp)
                return np.exp(-4 * fd) - ctrl_pen

    def _is_terminated(self):
        if self._torso_pos()[2] < 0.5:
            return True
        if self.phase == 3 and self._ball_kicked:
            if np.linalg.norm(self._ball_vel()) < BALL_STOP_VEL and \
               self._step_count > self._kick_step + 50:
                return True
        return False

    def render(self):
        if self._renderer is None:
            self._renderer = mujoco.Renderer(self.model, height=480, width=640)
        self._renderer.update_scene(self.data)
        return self._renderer.render()

    def close(self):
        if self._renderer:
            self._renderer.close()

print("HumanoidBallEnv ready  (obs_dim=%d, act_dim=%d)" % (OBS_DIM_SINGLE, N_ACT_SINGLE))


In [ ]:
import numpy as np
import mujoco
import gymnasium as gym
from gymnasium import spaces

N_ACT_SINGLE  = 20
OBS_DIM_SINGLE = 67
PASS_DISTANCE  = 3.0
BALL_STOP_VEL  = 0.05
HEIGHT_TARGET  = 1.25

class HumanoidBallEnv(gym.Env):
    """
    Single humanoid + ball.
    phase: 1=Stability  2=Touch  3=Pass
    """
    metadata = {"render_modes": ["rgb_array"], "render_fps": 30}

    def __init__(self, phase=1, render_mode=None):
        super().__init__()
        self.phase = phase
        self.render_mode = render_mode
        self.model = mujoco.MjModel.from_xml_path("models/humanoid_ball.xml")
        self.data  = mujoco.MjData(self.model)

        obs_hi = np.full(OBS_DIM_SINGLE, np.inf, dtype=np.float32)
        self.observation_space = spaces.Box(-obs_hi, obs_hi)
        self.action_space      = spaces.Box(-1.0, 1.0, shape=(N_ACT_SINGLE,), dtype=np.float32)

        self.max_steps   = 1000
        self._step_count = 0
        self._ball_kicked = False
        self._kick_step   = 0
        self._renderer    = None

        def bid(n): return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY,  n)
        def jid(n): return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, n)
        self._torso_id    = bid("torso")
        self._ball_id     = bid("ball")
        self._rfoot_id    = bid("right_foot")
        self._lfoot_id    = bid("left_foot")
        self._ball_jnt_id = jid("ball_joint")

    def _torso_pos(self): return self.data.xpos[self._torso_id].copy()
    def _ball_pos(self):  return self.data.xpos[self._ball_id].copy()
    def _ball_vel(self):
        va = self.model.jnt_dofadr[self._ball_jnt_id]
        return self.data.qvel[va:va+3].copy()

    def _get_obs(self):
        tp   = self._torso_pos()
        bp   = self._ball_pos()
        bv   = self._ball_vel()
        qpos = self.data.qpos.copy()
        qvel = self.data.qvel.copy()
        sens = self.data.sensordata.copy()
        obs  = np.concatenate([qpos, qvel, sens, bp-tp, bv]).astype(np.float32)
        if len(obs) < OBS_DIM_SINGLE: obs = np.pad(obs, (0, OBS_DIM_SINGLE-len(obs)))
        else: obs = obs[:OBS_DIM_SINGLE]
        return obs

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] += self.np_random.uniform(-0.01, 0.01, self.model.nq)
        ball_x = {1: 5.0, 2: 0.8, 3: 0.7}.get(self.phase, 0.7)
        qa = self.model.jnt_qposadr[self._ball_jnt_id]
        self.data.qpos[qa]   = ball_x
        self.data.qpos[qa+1] = 0.0
        self.data.qpos[qa+2] = 0.08
        self.data.qpos[qa+3:qa+7] = [1,0,0,0]
        mujoco.mj_forward(self.model, self.data)
        self._step_count  = 0
        self._ball_kicked = False
        self._kick_step   = 0
        return self._get_obs(), {}

    def step(self, action):
        self.data.ctrl[:] = np.clip(action, -1, 1)
        for _ in range(4): mujoco.mj_step(self.model, self.data)
        self._step_count += 1
        obs  = self._get_obs()
        rew  = self._compute_reward()
        term = self._is_terminated()
        trunc = self._step_count >= self.max_steps
        return obs, rew, term, trunc, {"phase": self.phase}

    def _compute_reward(self):
        tp = self._torso_pos(); bp = self._ball_pos(); bv = self._ball_vel()
        bspd = np.linalg.norm(bv)
        # --- Phase 1: Stability ---
        if self.phase == 1:
            h_err  = abs(tp[2] - HEIGHT_TARGET)
            upright = np.exp(-4*h_err)
            alive   = 1.0 if tp[2] > 0.8 else -2.0
            return alive + upright - 0.001*np.sum(self.data.ctrl**2)
        # --- Phase 2: Touch ---
        elif self.phase == 2:
            if tp[2] < 0.8: return -3.0
            rf = self.data.xpos[self._rfoot_id]; lf = self.data.xpos[self._lfoot_id]
            fd = min(np.linalg.norm(rf-bp), np.linalg.norm(lf-bp))
            approach = np.exp(-2*np.linalg.norm(tp[:2]-bp[:2]))
            touch    = np.exp(-5*fd)
            moved    = 2.0 if bspd > 0.3 else 0.0
            return 0.5*approach + touch + moved - 0.001*np.sum(self.data.ctrl**2)
        # --- Phase 3: Pass ---
        else:
            if tp[2] < 0.8: return -3.0
            if not self._ball_kicked and bspd > 1.0:
                self._ball_kicked = True; self._kick_step = self._step_count
            if self._ball_kicked:
                dist_err = abs(bp[0] - PASS_DISTANCE)
                dist_r   = np.exp(-1.5*dist_err)
                stop_r   = 5.0 if bspd < BALL_STOP_VEL else 0.0
                lat_pen  = 0.5*abs(bp[1])
                return dist_r + stop_r - lat_pen - 0.001*np.sum(self.data.ctrl**2)
            else:
                rf = self.data.xpos[self._rfoot_id]
                fd = np.linalg.norm(rf-bp)
                return np.exp(-4*fd) - 0.001*np.sum(self.data.ctrl**2)

    def _is_terminated(self):
        if self._torso_pos()[2] < 0.5: return True
        if self.phase == 3 and self._ball_kicked:
            bv = self._ball_vel()
            if np.linalg.norm(bv) < BALL_STOP_VEL and self._step_count > self._kick_step+50:
                return True
        return False

    def render(self):
        if self._renderer is None:
            self._renderer = mujoco.Renderer(self.model, height=480, width=640)
        self._renderer.update_scene(self.data)
        return self._renderer.render()

    def close(self):
        if self._renderer: self._renderer.close()

print("HumanoidBallEnv ready")


# Per-robot obs layout in TwoRobotPassEnv:
# torso_pos(3) + robot_qpos(27) + robot_qvel(26) + robot_sensors(8) + ball_rel_pos(3) + ball_vel(3) + active_flag(1) = 71
OBS_DIM_ROBOT = 71
N_ACT_TWO     = N_ACT_SINGLE * 2   # 40

# Two-robot qpos layout:  robot1[0:27]  robot2[27:54]  ball[54:61]
# Two-robot qvel layout:  robot1[0:26]  robot2[26:52]  ball[52:58]
# Two-robot sensors:      r1_accel(3)+r1_gyro(3)+r1_rfoot(1)+r1_lfoot(1)  = [0:8]
#                         r2_accel(3)+r2_gyro(3)+r2_rfoot(1)+r2_lfoot(1)  = [8:16]
#                         ball_touch(1) = [16]
R1_QPOS = slice(0,  27); R2_QPOS = slice(27, 54)
R1_QVEL = slice(0,  26); R2_QVEL = slice(26, 52)
R1_SENS = slice(0,   8); R2_SENS = slice(8,  16)

class TwoRobotPassEnv(gym.Env):
    """
    Two humanoid robots alternate passing the ball.
    Robot 1 (blue, x=-3) starts.  Robot 2 (red, x=+3) waits where the ball stops, then kicks back.
    Obs: 2 × OBS_DIM_ROBOT = 142    Act: 2 × N_ACT_SINGLE = 40
    """
    metadata = {"render_modes": ["rgb_array"], "render_fps": 30}

    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        self.model = mujoco.MjModel.from_xml_path("models/two_humanoids_ball.xml")
        self.data  = mujoco.MjData(self.model)

        obs_hi = np.full(OBS_DIM_ROBOT * 2, np.inf, dtype=np.float32)
        self.observation_space = spaces.Box(-obs_hi, obs_hi)
        self.action_space      = spaces.Box(-1.0, 1.0, shape=(N_ACT_TWO,), dtype=np.float32)

        def bid(n): return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY,  n)
        def jid(n): return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, n)
        self._r1t  = bid("robot1_torso")
        self._r2t  = bid("robot2_torso")
        self._ball = bid("ball")
        self._bjnt = jid("ball_joint")

        self.max_steps   = 2000
        self._step_count = 0
        self._active     = 1      # which robot kicks next
        self._ball_stop  = True   # ball is at rest
        self._pass_count = 0      # full round trips completed
        self._last_kick  = 0
        self._moving     = False
        self._renderer   = None

    def _bp(self): return self.data.xpos[self._ball].copy()
    def _bv(self):
        va = self.model.jnt_dofadr[self._bjnt]
        return self.data.qvel[va:va+3].copy()
    def _rp(self, r): return self.data.xpos[self._r1t if r == 1 else self._r2t].copy()

    def _robot_obs(self, robot_idx, active_flag):
        """Build a fixed-size obs vector for one robot."""
        rp   = self._rp(robot_idx)
        bp   = self._bp()
        bv   = self._bv()
        qp   = R1_QPOS if robot_idx == 1 else R2_QPOS
        qv   = R1_QVEL if robot_idx == 1 else R2_QVEL
        ss   = R1_SENS if robot_idx == 1 else R2_SENS
        arr  = np.concatenate([
            rp,
            self.data.qpos[qp],
            self.data.qvel[qv],
            self.data.sensordata[ss],
            bp - rp,
            bv,
            [float(active_flag)],
        ]).astype(np.float32)
        if len(arr) < OBS_DIM_ROBOT:
            arr = np.pad(arr, (0, OBS_DIM_ROBOT - len(arr)))
        return arr[:OBS_DIM_ROBOT]

    def _get_obs(self):
        o1 = self._robot_obs(1, self._active == 1)
        o2 = self._robot_obs(2, self._active == 2)
        return np.concatenate([o1, o2])

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] += self.np_random.uniform(-0.01, 0.01, self.model.nq)
        qa = self.model.jnt_qposadr[self._bjnt]
        self.data.qpos[qa]     = -2.0
        self.data.qpos[qa+1]   =  0.0
        self.data.qpos[qa+2]   =  0.08
        self.data.qpos[qa+3:qa+7] = [1, 0, 0, 0]
        mujoco.mj_forward(self.model, self.data)
        self._step_count = 0
        self._active     = 1
        self._ball_stop  = True
        self._pass_count = 0
        self._last_kick  = 0
        self._moving     = False
        return self._get_obs(), {}

    def step(self, action):
        self.data.ctrl[:] = np.clip(action, -1, 1)
        for _ in range(4): mujoco.mj_step(self.model, self.data)
        self._step_count += 1

        bv   = self._bv()
        bspd = np.linalg.norm(bv)

        # Detect kick start
        if not self._moving and bspd > 1.0:
            self._moving   = True
            self._last_kick = self._step_count
            self._ball_stop = False

        # Detect ball stopped → switch active robot
        if self._moving and bspd < BALL_STOP_VEL and self._step_count > self._last_kick + 50:
            self._ball_stop = True
            self._moving    = False
            self._active    = 2 if self._active == 1 else 1
            if self._active == 1:
                self._pass_count += 1  # full round trip

        obs   = self._get_obs()
        rew   = self._reward()
        r1a   = self._rp(1)[2] > 0.8
        r2a   = self._rp(2)[2] > 0.8
        term  = not (r1a and r2a)
        trunc = self._step_count >= self.max_steps
        return obs, rew, term, trunc, {"pass_count": self._pass_count, "active": self._active}

    def _reward(self):
        r1p  = self._rp(1)
        r2p  = self._rp(2)
        bp   = self._bp()
        bv   = self._bv()
        bspd = np.linalg.norm(bv)

        if r1p[2] < 0.8 or r2p[2] < 0.8:
            return -10.0

        ap  = r1p if self._active == 1 else r2p   # active robot
        tgt = r2p if self._active == 1 else r1p   # target robot

        if self._ball_stop:
            # Reward active robot for approaching the ball
            dr = np.exp(-2 * np.linalg.norm(ap[:2] - bp[:2]))
        else:
            # Reward for directing ball toward the target robot's x position
            de = abs(bp[0] - tgt[0])
            dr = np.exp(-1.5 * de) * 3.0
            if bspd < BALL_STOP_VEL:
                dr += 5.0 * np.exp(-2 * de)

        lat_pen  = 0.3 * abs(bp[1])
        ctrl_pen = 0.001 * np.sum(self.data.ctrl ** 2)
        return 1.0 + dr - lat_pen - ctrl_pen + self._pass_count * 2.0

    def render(self):
        if self._renderer is None:
            self._renderer = mujoco.Renderer(self.model, height=480, width=800)
        self._renderer.update_scene(self.data)
        return self._renderer.render()

    def close(self):
        if self._renderer:
            self._renderer.close()

print("TwoRobotPassEnv ready  (obs_dim=%d, act_dim=%d)" % (OBS_DIM_ROBOT * 2, N_ACT_TWO))


In [ ]:
class TwoRobotPassEnv(gym.Env):
    """
    Two humanoid robots alternately kick the ball to each other.
    Robot 1 (blue, left)  starts.
    Robot 2 (red,  right) waits where ball stops, then kicks back.
    Joint obs:  2 * OBS_DIM_SINGLE
    Joint act:  2 * N_ACT_SINGLE
    """
    metadata = {"render_modes": ["rgb_array"], "render_fps": 30}

    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        self.model = mujoco.MjModel.from_xml_path("models/two_humanoids_ball.xml")
        self.data  = mujoco.MjData(self.model)

        OD = OBS_DIM_SINGLE * 2
        obs_hi = np.full(OD, np.inf, dtype=np.float32)
        self.observation_space = spaces.Box(-obs_hi, obs_hi)
        self.action_space      = spaces.Box(-1.0, 1.0, shape=(N_ACT_SINGLE*2,), dtype=np.float32)

        def bid(n): return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY,  n)
        def jid(n): return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, n)
        self._r1t  = bid("robot1_torso");  self._r2t  = bid("robot2_torso")
        self._r1rf = bid("robot1_right_foot"); self._r2rf = bid("robot2_right_foot")
        self._ball  = bid("ball");  self._bjnt  = jid("ball_joint")
        self.max_steps   = 2000
        self._step_count = 0
        self._active     = 1
        self._ball_stop  = True
        self._pass_count = 0
        self._last_kick  = 0
        self._moving     = False
        self._renderer   = None

    def _bp(self): return self.data.xpos[self._ball].copy()
    def _bv(self):
        va = self.model.jnt_dofadr[self._bjnt]
        return self.data.qvel[va:va+3].copy()
    def _rp(self, r): return self.data.xpos[self._r1t if r==1 else self._r2t].copy()

    def _get_obs(self):
        r1p=self._rp(1); r2p=self._rp(2); bp=self._bp(); bv=self._bv()
        qpos=self.data.qpos.copy(); qvel=self.data.qvel.copy()
        sens=self.data.sensordata.copy()
        def mk(rp, qa, va, ss):
            arr = np.concatenate([rp, qpos[qa:qa+29], qvel[va:va+28],
                                   sens[ss:ss+4], bp-rp, bv,
                                   [float(self._active==(1 if qa==0 else 2))]]).astype(np.float32)
            if len(arr)<OBS_DIM_SINGLE: arr=np.pad(arr,(0,OBS_DIM_SINGLE-len(arr)))
            return arr[:OBS_DIM_SINGLE]
        o1 = mk(r1p, 0,  0, 0)
        o2 = mk(r2p, 29, 28, 4)
        return np.concatenate([o1, o2])

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] += self.np_random.uniform(-0.01,0.01,self.model.nq)
        qa = self.model.jnt_qposadr[self._bjnt]
        self.data.qpos[qa]=  -2.0
        self.data.qpos[qa+1]= 0.0
        self.data.qpos[qa+2]= 0.08
        self.data.qpos[qa+3:qa+7]=[1,0,0,0]
        mujoco.mj_forward(self.model,self.data)
        self._step_count=0; self._active=1; self._ball_stop=True
        self._pass_count=0; self._last_kick=0; self._moving=False
        return self._get_obs(), {}

    def step(self, action):
        self.data.ctrl[:]=np.clip(action,-1,1)
        for _ in range(4): mujoco.mj_step(self.model,self.data)
        self._step_count+=1
        bv=self._bv(); bspd=np.linalg.norm(bv)
        if not self._moving and bspd>1.0:
            self._moving=True; self._last_kick=self._step_count; self._ball_stop=False
        if self._moving and bspd<BALL_STOP_VEL and self._step_count>self._last_kick+50:
            self._ball_stop=True; self._moving=False
            self._active = 2 if self._active==1 else 1
            if self._active==1: self._pass_count+=1
        obs=self._get_obs(); rew=self._reward()
        r1a=self._rp(1)[2]>0.8; r2a=self._rp(2)[2]>0.8
        term=not(r1a and r2a); trunc=self._step_count>=self.max_steps
        return obs,rew,term,trunc,{"pass_count":self._pass_count,"active":self._active}

    def _reward(self):
        r1p=self._rp(1); r2p=self._rp(2); bp=self._bp(); bv=self._bv()
        bspd=np.linalg.norm(bv)
        if r1p[2]<0.8 or r2p[2]<0.8: return -10.0
        ap  = r1p if self._active==1 else r2p
        tgt = r2p if self._active==1 else r1p
        if self._ball_stop:
            dr = np.exp(-2*np.linalg.norm(ap[:2]-bp[:2]))
        else:
            de = abs(bp[0]-tgt[0])
            dr = np.exp(-1.5*de)*3.0
            if bspd<BALL_STOP_VEL: dr += 5.0*np.exp(-2*de)
        lat = 0.3*abs(bp[1])
        cc  = 0.001*np.sum(self.data.ctrl**2)
        return 1.0 + dr - lat - cc + self._pass_count*2.0

    def render(self):
        if self._renderer is None:
            self._renderer = mujoco.Renderer(self.model,height=480,width=800)
        self._renderer.update_scene(self.data)
        return self._renderer.render()

    def close(self):
        if self._renderer: self._renderer.close()

print("TwoRobotPassEnv ready")


## 📈 Cell 6 – Curriculum Callback

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
import time

class CurriculumCallback(BaseCallback):
    """Advances phase when rolling mean reward crosses threshold."""
    THRESHOLDS = {1: 300, 2: 250}   # raw reward thresholds (tunable)
    NAMES      = {1:"STABILITY", 2:"TOUCH", 3:"PASS"}

    def __init__(self, vec_env, max_phase=3, window=20, verbose=1):
        super().__init__(verbose)
        self.vec_env   = vec_env
        self.max_phase = max_phase
        self.window    = window
        self._phase    = 1
        self._buf      = []

    @property
    def phase(self): return self._phase

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self._buf.append(info["episode"]["r"])
        if len(self._buf) >= self.window:
            mean_r = np.mean(self._buf[-self.window:])
            thresh = self.THRESHOLDS.get(self._phase)
            if thresh and self._phase < self.max_phase and mean_r > thresh:
                self._phase += 1
                print(f"\n>>> Curriculum: Phase {self._phase} ({self.NAMES.get(self._phase, '?')})  mean_r={mean_r:.1f}")
                for e in self.vec_env.envs:
                    e.env.phase = self._phase
                self._buf.clear()
        return True

print("CurriculumCallback ready")


## ✅ Cell 7 – Sanity Check (run first!)

In [ ]:
# Quick sanity check – run this BEFORE long training
env = HumanoidBallEnv(phase=1)
obs,_ = env.reset()
print(f"Single  obs={obs.shape}  act={env.action_space.shape}")
for ph in [1,2,3]:
    env.phase=ph; obs,_=env.reset(); r=0
    for _ in range(50):
        o,rew,t,tr,_ = env.step(env.action_space.sample()); r+=rew
        if t or tr: break
    print(f"  Phase {ph}  50-step reward={r:.2f}")
env.close()

env2 = TwoRobotPassEnv()
obs2,_=env2.reset(); r2=0
print(f"\nTwo-robot obs={obs2.shape}  act={env2.action_space.shape}")
for _ in range(50):
    o,rew,t,tr,info=env2.step(env2.action_space.sample()); r2+=rew
    if t or tr: break
print(f"  50-step reward={r2:.2f}  passes={info['pass_count']}")
env2.close()
print("\nAll checks passed!")


## 🚀 Cell 8 – Train Single Robot (Phases 1→3)

In [ ]:
import os; os.makedirs("checkpoints",exist_ok=True); os.makedirs("videos",exist_ok=True)

def make_single(phase=1):
    def _f():
        env = HumanoidBallEnv(phase=phase)
        return Monitor(env)
    return _f

N_ENVS = 4   # reduce to 2 if OOM

vec_env = DummyVecEnv([make_single(1)] * N_ENVS)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10., clip_reward=10.)

model = PPO(
    "MlpPolicy", vec_env,
    learning_rate=3e-4, n_steps=2048, batch_size=256,
    n_epochs=10, gamma=0.99, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=[512,512,256]),
    verbose=1, device="auto", tensorboard_log="./tb/"
)

cb = CurriculumCallback(vec_env=vec_env, max_phase=3, window=20)

TOTAL = 5_000_000   # ~3-5 h on T4; use 100_000 for quick test
print(f"Training single-robot curriculum  {TOTAL:,} steps ...")
model.learn(TOTAL, callback=cb, progress_bar=True)
model.save("checkpoints/single_final")
vec_env.save("checkpoints/vecnorm_single.pkl")
print("Done. Final phase:", cb.phase)


## 🤝 Cell 9 – Train Two Robots (Phase 4)

In [ ]:
def make_two():
    def _f(): return Monitor(TwoRobotPassEnv())
    return _f

two_vec = DummyVecEnv([make_two()] * 2)
two_vec = VecNormalize(two_vec, norm_obs=True, norm_reward=True, clip_obs=10., clip_reward=10.)

two_model = PPO(
    "MlpPolicy", two_vec,
    learning_rate=1e-4, n_steps=2048, batch_size=256,
    n_epochs=10, gamma=0.99, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.005, vf_coef=0.5, max_grad_norm=0.5,
    policy_kwargs=dict(net_arch=[512,512,256]),
    verbose=1, device="auto", tensorboard_log="./tb/"
)

TWO_STEPS = 3_000_000
print(f"Training two-robot passing  {TWO_STEPS:,} steps ...")
two_model.learn(TWO_STEPS, progress_bar=True)
two_model.save("checkpoints/two_final")
two_vec.save("checkpoints/vecnorm_two.pkl")
print("Two-robot training done!")


import imageio
import mediapy as media
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

def record_episode(model, env_cls, env_kwargs, norm_path, out_path, n_steps=600):
    """Record a single episode to an MP4.  Pass the loaded PPO model (not a path)."""
    env   = env_cls(**env_kwargs)
    dummy = DummyVecEnv([lambda: env])
    norm  = VecNormalize.load(norm_path, dummy)
    norm.training    = False
    norm.norm_reward = False

    obs, _  = env.reset()
    frames  = []
    for _ in range(n_steps):
        obs_n = norm.normalize_obs(obs[np.newaxis])[0]
        act, _ = model.predict(obs_n, deterministic=True)
        obs, _, term, trunc, _ = env.step(act)
        frames.append(env.render())
        if term or trunc:
            break
    env.close()
    imageio.mimsave(out_path, frames, fps=30)
    print(f"Saved {out_path}  ({len(frames)} frames)")
    return frames


# --- Phase 3 single-robot ---
single_model = PPO.load("checkpoints/single_final")
f1 = record_episode(
    single_model, HumanoidBallEnv, {"phase": 3},
    "checkpoints/vecnorm_single.pkl",
    "videos/phase3_pass.mp4", 800,
)
media.show_video(f1, fps=30)

# --- Phase 4 two-robot ---
two_model = PPO.load("checkpoints/two_final")
f2 = record_episode(
    two_model, TwoRobotPassEnv, {},
    "checkpoints/vecnorm_two.pkl",
    "videos/two_pass.mp4", 1500,
)
media.show_video(f2, fps=30)


In [ ]:
import imageio, mediapy as media

def record_episode(model_path, env_cls, env_kwargs, norm_path, out_path, n_steps=600):
    env = env_cls(**env_kwargs)
    dummy = DummyVecEnv([lambda: env])
    norm  = VecNormalize.load(norm_path, dummy)
    norm.training=False; norm.norm_reward=False
    obs,_= env.reset(); frames=[]
    for _ in range(n_steps):
        on = norm.normalize_obs(obs[np.newaxis])[0]
        act,_ = model.predict(on, deterministic=True)
        obs,_,term,trunc,_ = env.step(act)
        frames.append(env.render())
        if term or trunc: break
    env.close()
    imageio.mimsave(out_path, frames, fps=30)
    print(f"Saved {out_path}  ({len(frames)} frames)")
    return frames

model = PPO.load("checkpoints/single_final")
f1 = record_episode(None, HumanoidBallEnv, {"phase":3},
                    "checkpoints/vecnorm_single.pkl",
                    "videos/phase3_pass.mp4", 800)
media.show_video(f1, fps=30)

two_model_r = PPO.load("checkpoints/two_final")
f2 = record_episode(None, TwoRobotPassEnv, {},
                    "checkpoints/vecnorm_two.pkl",
                    "videos/two_pass.mp4", 1500)
media.show_video(f2, fps=30)


## 📊 Cell 11 – TensorBoard

In [ ]:
# Optional: view training curves in TensorBoard
%load_ext tensorboard
%tensorboard --logdir ./tb/


## 📋 Architecture Summary

| Component | Details |
|-----------|----------|
| Physics | MuJoCo 3.x, 5 ms timestep, Newton solver |
| Robot | 20-DoF humanoid (hips, knees, ankles, shoulders, elbows, abdomen) |
| Ball | 0.15 kg sphere, realistic friction & bounce |
| Algorithm | PPO [512, 512, 256] MLP, VecNormalize |
| Curriculum | Phase 1→2→3 on rolling-mean reward threshold |
| Phase 4 | Joint 40-dim action, alternating-turn reward |

### Tips
- **Cell 7** sanity check first — catches XML / import errors quickly
- **Quick run**: change `TOTAL = 100_000` in Cell 8
- Full single-robot training ~3-5 h on T4 GPU
- `N_ENVS = 4` parallel envs; lower to 2 if GPU OOM
- Transfer learning: Cell 9 starts from scratch but you can load single-robot weights into overlapping layers manually